In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction](introyt1_tutorial.html) \|\|
[Tensors](tensors_deeper_tutorial.html) \|\|
[Autograd](autogradyt_tutorial.html) \|\| [Building
Models](modelsyt_tutorial.html) \|\| [TensorBoard
Support](tensorboardyt_tutorial.html) \|\| **Training Models** \|\|
[Model Understanding](captumyt.html)

Training with PyTorch
=====================

Follow along with the video below or on
[youtube](https://www.youtube.com/watch?v=jF43_wj_DCQ).



In [ ]:
# Run this cell to load the video
from IPython.display import display, HTML
html_code = """
<div style="margin-top:10px; margin-bottom:10px;">
  <iframe width="560" height="315" src="https://www.youtube.com/embed/jF43_wj_DCQ" frameborder="0" allow="accelerometer; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
</div>
"""
display(HTML(html_code))

Introduction  
简介
------------

In past videos, we've discussed and demonstrated:  
在之前的视频中，我们讨论并演示了以下内容：

-   Building models with the neural network layers and functions of the
    torch.nn module  
    使用 `torch.nn` 模块中的神经网络层和函数构建模型
-   The mechanics of automated gradient computation, which is central to
    gradient-based model training  
    自动梯度计算的机制，这是基于梯度的模型训练的核心
-   Using TensorBoard to visualize training progress and other
    activities  
    使用 TensorBoard 可视化训练进度和其他活动

In this video, we'll be adding some new tools to your inventory:  
在本视频中，我们将为你的工具箱添加一些新工具：

-   We'll get familiar with the dataset and dataloader abstractions, and
    how they ease the process of feeding data to your model during a
    training loop  
    我们将熟悉 `Dataset`（数据集）和 `DataLoader`（数据加载器）的抽象概念，以及它们如何在训练循环中简化向模型输入数据的过程
-   We'll discuss specific loss functions and when to use them  
    我们将讨论特定的损失函数及其适用场景
-   We'll look at PyTorch optimizers, which implement algorithms to
    adjust model weights based on the outcome of a loss function  
    我们将了解 PyTorch 优化器，它们实现了根据损失函数的结果来调整模型权重的算法

Finally, we'll pull all of these together and see a full PyTorch
training loop in action.    
最后，我们将把这些工具整合起来，完整地演示一个 PyTorch 训练循环的实际运行。

Dataset and DataLoader  
Dataset 和 DataLoader
----------------------

The `Dataset` and `DataLoader` classes encapsulate the process of
pulling your data from storage and exposing it to your training loop in
batches.  
`Dataset` 和 `DataLoader` 类封装了从存储中提取数据并以批次（batches）形式暴露给训练循环的过程。

The `Dataset` is responsible for accessing and processing single
instances of data.  
`Dataset` 负责访问和处理单个数据样本。

The `DataLoader` pulls instances of data from the `Dataset` (either
automatically or with a sampler that you define), collects them in
batches, and returns them for consumption by your training loop. The
`DataLoader` works with all kinds of datasets, regardless of the type of
data they contain.  
`DataLoader` 从 `Dataset` 中提取数据样本（可以自动提取，也可以通过你定义的采样器提取），将它们收集成批次，并返回给训练循环使用。无论包含何种类型的数据，`DataLoader` 都能与各种数据集配合使用。

For this tutorial, we'll be using the Fashion-MNIST dataset provided by
TorchVision. We use `torchvision.transforms.v2.Normalize()` to
zero-center and normalize the distribution of the image tile content,
and download both training and validation data splits.  
在本教程中，我们将使用 TorchVision 提供的 Fashion-MNIST 数据集。我们使用 `torchvision.transforms.v2.Normalize()` 对图像内容进行零中心化（zero-center）和归一化分布处理，并下载训练集和验证集的数据划分。

In [ ]:
import torch
import torchvision
from torchvision.transforms import v2

# PyTorch TensorBoard support
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime


transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5,), (0.5,))
])

# Create datasets for training & validation, download if necessary
training_set = torchvision.datasets.FashionMNIST('./data', train=True, transform=transform, download=True)
validation_set = torchvision.datasets.FashionMNIST('./data', train=False, transform=transform, download=True)

# Create data loaders for our datasets; shuffle for training, not for validation
training_loader = torch.utils.data.DataLoader(training_set, batch_size=4, shuffle=True)
validation_loader = torch.utils.data.DataLoader(validation_set, batch_size=4, shuffle=False)

# Class labels
classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
        'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot')

# Report split sizes
print(f'Training set has {len(training_set)} instances')
print(f'Validation set has {len(validation_set)} instances')

As always, let's visualize the data as a sanity check:  
一如既往，让我们先可视化数据，以此作为正确性的检查：

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Helper function for inline image display
def matplotlib_imshow(img, one_channel=False):
    if one_channel:
        img = img.mean(dim=0)
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    if one_channel:
        plt.imshow(npimg, cmap="Greys")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))

dataiter = iter(training_loader)
images, labels = next(dataiter)

# Create a grid from the images and show them
img_grid = torchvision.utils.make_grid(images)
matplotlib_imshow(img_grid, one_channel=True)
print('  '.join(classes[labels[j]] for j in range(4)))

The Model  
模型
=========

The model we'll use in this example is a variant of LeNet-5 - it should
be familiar if you've watched the previous videos in this series.  
我们在本例中使用的模型是 LeNet-5 的一个变体。如果你看过本系列的前面视频，应该对它很熟悉。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# PyTorch models inherit from torch.nn.Module
class GarmentClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    

model = GarmentClassifier()

Loss Function  
损失函数
=============

For this example, we'll be using a cross-entropy loss. For demonstration
purposes, we'll create batches of dummy output and label values, run
them through the loss function, and examine the result.  
对于本例，我们将使用交叉熵损失（cross-entropy loss）。为了演示，我们将创建一些虚拟的输出和标签值批次，将它们传入损失函数，并检查结果。

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

# NB: Loss functions expect data in batches, so we're creating batches of 4
# Represents the model's confidence in each of the 10 classes for a given input
dummy_outputs = torch.rand(4, 10)
# Represents the correct class among the 10 being tested
dummy_labels = torch.tensor([1, 5, 3, 7])
    
print(dummy_outputs)
print(dummy_labels)

loss = loss_fn(dummy_outputs, dummy_labels)
print(f'Total loss for this batch: {loss.item()}')

Optimizer  
优化器
=========

For this example, we'll be using simple [stochastic gradient
descent](https://pytorch.org/docs/stable/optim.html) with momentum.  
对于本例，我们将使用带有动量（momentum）的简单[随机梯度下降（SGD）](https://pytorch.org/docs/stable/optim.html)。

It can be instructive to try some variations on this optimization
scheme:  
尝试对该优化方案进行一些调整会很有启发性：

-   Learning rate determines the size of the steps the optimizer takes.
    What does a different learning rate do to the your training results,
    in terms of accuracy and convergence time?  
    **学习率（Learning rate）**决定了优化器迈出的步子大小。不同的学习率会对你的训练结果产生什么影响，尤其是在准确性和收敛时间方面？
-   Momentum nudges the optimizer in the direction of strongest gradient
    over multiple steps. What does changing this value do to your
    results?  
    **动量（Momentum）**会推动优化器沿着多个步骤中梯度最强的方向前进。改变这个值会对你的结果产生什么影响？
-   Try some different optimization algorithms, such as averaged SGD,
    Adagrad, or Adam. How do your results differ?  
    尝试一些不同的优化算法，例如平均SGD（averaged SGD）、Adagrad 或 Adam。你的结果会有什么不同？


In [ ]:
# Optimizers specified in the torch.optim package
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

The Training Loop  
训练循环
=================

Below, we have a function that performs one training epoch. It
enumerates data from the DataLoader, and on each pass of the loop does
the following:  
下面，我们定义了一个执行单个训练 epoch（轮次）的函数。它从 DataLoader 中枚举数据，并在循环的每次迭代中执行以下操作：


-   Gets a batch of training data from the DataLoader  
    从 DataLoader 获取一个批次的训练数据
-   Zeros the optimizer's gradients  
    将优化器的梯度清零
-   Performs an inference - that is, gets predictions from the model for
    an input batch  
    执行前向推理——即获取模型对该输入批次的预测结果
-   Calculates the loss for that set of predictions vs. the labels on
    the dataset  
    计算该组预测结果与数据集中真实标签之间的损失
-   Calculates the backward gradients over the learning weights  
    针对学习权重计算反向传播梯度
-   Tells the optimizer to perform one learning step - that is, adjust
    the model's learning weights based on the observed gradients for
    this batch, according to the optimization algorithm we chose  
    指示优化器执行一次学习步骤——即根据我们选择的优化算法，基于该批次观察到的梯度来调整模型的学习权重
-   It reports on the loss for every 1000 batches.  
    每处理 1000 个批次报告一次损失
-   Finally, it reports the average per-batch loss for the last 1000
    batches, for comparison with a validation run  
    最后，报告最后 1000 个批次的平均单批次损失，以便与验证运行进行对比


In [ ]:
def train_one_epoch(epoch_index, tb_writer):
    running_loss = 0.
    last_loss = 0.
    
    # Here, we use enumerate(training_loader) instead of
    # iter(training_loader) so that we can track the batch
    # index and do some intra-epoch reporting
    for i, data in enumerate(training_loader):
        # Every data instance is an input + label pair
        inputs, labels = data
        
        # Zero your gradients for every batch!
        optimizer.zero_grad()
        
        # Make predictions for this batch
        outputs = model(inputs)
        
        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels)
        loss.backward()
        
        # Adjust learning weights
        optimizer.step()
        
        # Gather data and report
        running_loss += loss.item()
        if i % 1000 == 999:
            last_loss = running_loss / 1000 # loss per batch
            print(f'  batch {i + 1} loss: {last_loss}')
            tb_x = epoch_index * len(training_loader) + i + 1
            tb_writer.add_scalar('Loss/train', last_loss, tb_x)
            running_loss = 0.
            
    return last_loss

Per-Epoch Activity  
每个 Epoch 的常规操作
==================

There are a couple of things we'll want to do once per epoch:  
在每个 epoch 结束时，我们通常需要执行以下几项操作：

-   Perform validation by checking our relative loss on a set of data
    that was not used for training, and report this  
    在未用于训练的数据集上检查相对损失来进行验证，并报告该结果
-   Save a copy of the model  
    保存一份模型的副本

Here, we'll do our reporting in TensorBoard. This will require going to
the command line to start TensorBoard, and opening it in another browser
tab.  
在这里，我们将通过 TensorBoard 来输出这些报告。这需要在命令行中启动 TensorBoard，并在另一个浏览器标签页中打开它。


In [ ]:
# Initializing in a separate cell so we can easily add more epochs to the same run
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(f'runs/fashion_trainer_{timestamp}')
epoch_number = 0

EPOCHS = 5

best_vloss = 1_000_000.

for epoch in range(EPOCHS):
    print(f'EPOCH {epoch_number + 1}:')
    
    # Make sure gradient tracking is on, and do a pass over the data
    model.train(True)
    avg_loss = train_one_epoch(epoch_number, writer)
    

    running_vloss = 0.0
    # Set the model to evaluation mode, disabling dropout and using population 
    # statistics for batch normalization.
    model.eval()

    # Disable gradient computation and reduce memory consumption.
    with torch.no_grad():
        for i, vdata in enumerate(validation_loader):
            vinputs, vlabels = vdata
            voutputs = model(vinputs)
            vloss = loss_fn(voutputs, vlabels)
            running_vloss += vloss
    
    avg_vloss = running_vloss / (i + 1)
    print(f'LOSS train {avg_loss} valid {avg_vloss}')
    
    # Log the running loss averaged per batch
    # for both training and validation
    writer.add_scalars('Training vs. Validation Loss',
                    { 'Training' : avg_loss, 'Validation' : avg_vloss },
                    epoch_number + 1)
    writer.flush()
    
    # Track best performance, and save the model's state
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = f'model_{timestamp}_{epoch_number}'
        torch.save(model.state_dict(), model_path)
    
    epoch_number += 1

To load a saved version of the model:  
加载已保存的模型版本：

``` {.python}
saved_model = GarmentClassifier()
saved_model.load_state_dict(torch.load(PATH))
```

Once you've loaded the model, it's ready for whatever you need it for
-more training, inference, or analysis.  
加载模型后，它就可以随时用于你需要的任何用途——无论是继续训练、推理还是分析。

Note that if your model has constructor parameters that affect model
structure, you'll need to provide them and configure the model
identically to the state in which it was saved.  
请注意，如果你的模型具有影响模型结构的构造函数参数，你需要提供这些参数，并将模型配置为与保存时完全相同的状态。

Other Resources  
其他资源
===============

-   Docs on the [data
    utilities](https://pytorch.org/docs/stable/data.html), including
    Dataset and DataLoader, at pytorch.org  
    **数据工具文档**：包括 pytorch.org 上的 Dataset 和 DataLoader 相关文档。
-   A [note on the use of pinned
    memory](https://pytorch.org/docs/stable/notes/cuda.html#cuda-memory-pinning)
    for GPU training  
    **固定内存（Pinned Memory）说明**：关于在 GPU 训练中使用固定内存的注意事项。
-   Documentation on the datasets available in
    [TorchVision](https://pytorch.org/vision/stable/datasets.html),
    [TorchText](https://pytorch.org/text/stable/datasets.html), and
    [TorchAudio](https://pytorch.org/audio/stable/datasets.html)  
    **数据集文档**：涵盖 TorchVision、TorchText 和 TorchAudio 中可用数据集的文档。
-   Documentation on the [loss
    functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
    available in PyTorch  
    **损失函数文档**：PyTorch 中可用的损失函数相关文档。
-   Documentation on the [torch.optim
    package](https://pytorch.org/docs/stable/optim.html), which includes
    optimizers and related tools, such as learning rate scheduling  
    **torch.optim 包文档**：包含优化器及相关工具（如学习率调度）的文档。
-   A detailed [tutorial on saving and loading
    models](https://pytorch.org/tutorials/beginner/saving_loading_models.html)  
    **模型保存与加载详细教程**：关于如何保存和加载模型的专门教程。
-   The [Tutorials section of
    pytorch.org](https://pytorch.org/tutorials/) contains tutorials on a
    broad variety of training tasks, including classification in
    different domains, generative adversarial networks, reinforcement
    learning, and more  
    **pytorch.org 教程专区**：包含各种训练任务的教程，涵盖不同领域的分类、生成对抗网络（GANs）、强化学习等。
